In [10]:
import time
import logging
import signal
import sys
import gc
from datetime import datetime
from typing import Optional

import config
from data.manager import DataManager
from engine.scanner_engine import ScannerEngine
from reports.excel_exporter import ExcelExporter
from reports.chart_plotter import ChartPlotter
from scoring.ranker import OpportunityRanker, RankingProfile
from strategies.core import _load_strategies
from analytics.risk_engine import RiskEngine
from analytics.strategy_classifier import StrategyClassifier
from filters.strategy_filters import apply_strategy_filter

logger = logging.getLogger("OptionScanner.Main")

In [8]:
class OptionScanner:
    """کلاس اصلی اسکنر با پشتیبانی از معماری V5 و بهینه‌سازی حافظه"""

    __slots__ = (
        'interval_minutes', 'parallel', 'max_workers', 'max_cycles',
        'is_running', 'cycle_count', 'data_manager', 'ranker',
        'excel_exporter', 'chart_plotter', '_last_snapshot'
    )

    def __init__(
            self,
            interval_minutes: Optional[int] = None,
            parallel: Optional[bool] = None,
            max_workers: Optional[int] = None,
            max_cycles: Optional[int] = None):

        sys_config = config.get_system_config()

        self.interval_minutes = interval_minutes or sys_config.get("scan_interval_minutes", 3)
        self.parallel = parallel if parallel is not None else sys_config.get("parallel_enabled", True)
        self.max_workers = max_workers or sys_config.get("max_workers", 1)
        

        cfg_max = sys_config.get("max_cycles", 0) or 0
        self.max_cycles = max_cycles if max_cycles is not None else cfg_max

        self.is_running = True
        self.cycle_count = 0

        logger.info("Loading strategies definitions...")
        _load_strategies()

        self.data_manager = DataManager(
            cache_dir=str(config.CACHE_DIR),
            use_cache=True,
            ttl_seconds=config.CACHE_TTL_SECONDS
        )

        profile_map = {
            "conservative": RankingProfile.CONSERVATIVE,
            "balanced": RankingProfile.BALANCED,
            "aggressive": RankingProfile.AGGRESSIVE,
            "income": RankingProfile.INCOME,
            "volatility": RankingProfile.VOLATILITY,
        }
        profile_name = config.RANKING_CONFIG.get("default_profile", "balanced")
        profile = profile_map.get(profile_name, RankingProfile.BALANCED)

        self.ranker = OpportunityRanker(default_profile=profile)
        self.excel_exporter = ExcelExporter(output_dir=str(config.OUTPUT_DIR))
        self.chart_plotter = ChartPlotter(output_dir=str(config.CHARTS_DIR))

        signal.signal(signal.SIGINT, self._signal_handler)
        signal.signal(signal.SIGTERM, self._signal_handler)

        self._last_snapshot = None

    def _signal_handler(self, signum, frame) -> None:
        """مدیریت ایمن سیگنال‌های خروج"""
        signal_name = "SIGINT" if signum == signal.SIGINT else "SIGTERM"
        logger.info(f"Received {signal_name}. Graceful shutdown sequence initiated...")
        self.is_running = False
    
    def run_cycle(self) -> bool:
        calc_advanced = config.FEATURE_FLAGS.get("calculate_greeks", True)
        snapshot = self.data_manager.get_market_snapshot(force_refresh=False, calc_advanced=calc_advanced)
        self._last_snapshot = snapshot
        return True

    @property
    def last_snapshot(self):
        """دسترسی به آخرین snapshot"""
        return self._last_snapshot

scanner = OptionScanner()
scanner.run_cycle()
snapshot = scanner.last_snapshot
engine = ScannerEngine(snapshot=snapshot)

In [11]:
scan_result = engine.execute_full_scan()
scan_result

ScanResult(timestamp=datetime.datetime(2026, 7, 10, 9, 7, 23, 887859), total_strategies_scanned=1, total_combinations_generated=1, total_combinations_filtered=0, candidates=[], opportunities=[Opportunity(strategy_name='covered_call', underlying_ticker='اهرم', legs=[LegDefinition(side=<Side.BUY: 'BUY'>, ratio=1, contract=OptionContract(ticker='اهرم', name='اختيارخ اهرم-22000-1405/04/31', underlying_ticker='اهرم', option_type=<OptionType.STOCK: 0>, strike_price=46366.0, contract_size=1, expiry_date=None, days_to_maturity=0, bid=0.0, ask=0.0, last_price=46366.0, close_price=0.0, underlying_price=46366.0, yesterday_price=0.0, volume=0, open_interest=0, value=0.0, bid_volume=0, ask_volume=0, initial_margin=0.0, iv=None, delta=None, gamma=None, theta=None, vega=None, rho=None, implied_volatility=None, iv_hv_ratio=1.0, instrument_code='', instrument_code_ua=''), entry_price=46366.0), LegDefinition(side=<Side.BUY: 'BUY'>, ratio=1, contract=OptionContract(ticker='اهرم', name='اختيارخ اهرم-22000